# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [1]:
def transposee(A):
    return [[A[i][j] for i in range(len(A))] for j in range(len(A[0]))]

print(transposee([[1, 2, 3], [4, 5, 6]]))

[[1, 4], [2, 5], [3, 6]]


## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [2]:
def produit(A, B):
    """Produit matriciel A x B, en Python pur."""
    n, p = len(A), len(A[0])
    if p != len(B):
        raise ValueError("Le nombre de colonnes de A doit être égal au nombre de lignes de B.")
    m = len(B[0])
    C = [[0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            for k in range(p):
                C[i][j] += A[i][k] * B[k][j]
    return C

identite = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print(produit(A, identite) == A)
print(produit([[1, 2], [3, 4]], [[0, 1], [1, 0]]))

True
[[2, 1], [4, 3]]


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
import random
import time

random.seed(42)

def matrice_aleatoire(n):
    """Matrice carrée n x n de flottants aléatoires dans [0, 1[."""
    return [[random.random() for _ in range(n)] for _ in range(n)]

temps = {}
for n in (40, 80, 120):
    A = matrice_aleatoire(n)
    B = matrice_aleatoire(n)
    debut = time.perf_counter()
    produit(A, B)
    temps[n] = time.perf_counter() - debut
    print(f"n = {n:>3} : {temps[n]:.4f} s")

print(f"rapport 80 / 40  : {temps[80] / temps[40]:.1f}")
print(f"rapport 120 / 40 : {temps[120] / temps[40]:.1f}")

# Trois boucles imbriquées de longueur n font n^3 multiplications :
# quand la taille double, le temps est multiplié par 2^3 = 8 (et par 3^3 = 27 quand elle triple). 

n =  40 : 0.0026 s
n =  80 : 0.0199 s
n = 120 : 0.0662 s
rapport 80 / 40  : 7.7
rapport 120 / 40 : 25.7


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [4]:
import numpy as np

n = 120
A = matrice_aleatoire(n)
B = matrice_aleatoire(n)

debut = time.perf_counter()
C = produit(A, B)
t_python = time.perf_counter() - debut

An, Bn = np.array(A), np.array(B)
An @ Bn  # premier appel à blanc : il paie l'initialisation de numpy, pas le calcul
debut = time.perf_counter()
Cn = An @ Bn
t_numpy = time.perf_counter() - debut

print(f"python pur : {t_python:.4f} s")
print(f"numpy      : {t_numpy:.6f} s")
print(f"rapport    : {t_python / t_numpy:.0f}")
print(np.allclose(C, Cn))  # les deux produits donnent bien le même résultat

python pur : 0.0657 s
numpy      : 0.000068 s
rapport    : 968
True


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
def resoudre(A, b, tolerance=1e-14):
    """Résout A x = b par le pivot de Gauss avec pivot partiel.

    Ni A ni b ne sont modifiés. Un pivot dont la valeur absolue est
    inférieure ou égale à tolerance est considéré comme nul.
    """
    n = len(A)
    M = [list(A[i]) + [b[i]] for i in range(n)]

    for k in range(n):
        p = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[p][k]) <= tolerance:
            raise ValueError("Système singulier.")
        M[k], M[p] = M[p], M[k]

        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] -= facteur * M[k][j]

    x = [0] * n
    for i in range(n - 1, -1, -1):
        somme = M[i][n]
        for j in range(i + 1, n):
            somme -= M[i][j] * x[j]
        x[i] = somme / M[i][i]
    return x

A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
b = [2, -1, 8]
x = resoudre(A, b)
print([round(v, 10) for v in x])

Ax = [sum(A[i][j] * x[j] for j in range(3)) for i in range(3)]
print([round(v, 10) for v in Ax])

[3.0, -1.0, 0.5]
[2.0, -1.0, 8.0]


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [6]:
random.seed(7)
n = 50
A = matrice_aleatoire(n)
b = [random.random() for _ in range(n)]

x_gauss = resoudre(A, b)
x_numpy = np.linalg.solve(np.array(A), np.array(b))

ecart = max(abs(x_gauss[i] - x_numpy[i]) for i in range(n))
print("écart maximal :", ecart)
print(ecart < 1e-9)

écart maximal : 2.984279490192421e-13
True


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [7]:
def hilbert(n):
    return np.array([[1.0 / (i + j + 1) for j in range(n)] for i in range(n)])

n = 12
H = hilbert(n)
x_vrai = np.ones(n)          # on choisit la solution d'avance
b = H @ x_vrai               # et on fabrique le second membre correspondant

x_solve = np.linalg.solve(H, b)
x_inverse = np.linalg.inv(H) @ b

print("erreur avec solve   :", float(np.max(np.abs(x_solve - x_vrai))))
print("erreur avec inv @ b :", float(np.max(np.abs(x_inverse - x_vrai))))
print("conditionnement     : {:.3e}".format(float(np.linalg.cond(H))))

erreur avec solve   : 0.6961070342706484
erreur avec inv @ b : 21.0
conditionnement     : 1.808e+16


**Commentaire.** La vraie solution ne contient que des 1. `solve` se trompe d'environ 0,7 et `inv(H) @ b` d'environ 21 : `solve` est la plus proche, à peu près trente fois meilleure, mais aucune des deux n'est utilisable.

Le conditionnement vaut environ $1{,}8 \times 10^{16}$. Il mesure de combien une petite perturbation des données (ici les arrondis des flottants, de l'ordre de $10^{-16}$) peut être amplifiée dans la solution. À $10^{16}$, l'amplification mange les 16 chiffres significatifs d'un flottant : plus aucun chiffre du résultat n'est fiable. C'est le problème qui est mal posé, pas l'algorithme.

Avec `n = 10`, le conditionnement tombe à $1{,}6 \times 10^{13}$ et les erreurs à $1{,}7 \times 10^{-4}$ (`solve`) et $5{,}5 \times 10^{-3}$ (`inv`) : le rapport de trente entre les deux méthodes reste le même. D'où la règle : on écrit `np.linalg.solve(A, b)`, jamais `np.linalg.inv(A) @ b`.

## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [8]:
def rotation(angle):
    """Matrice de la rotation d'angle donné (en radians) autour de l'origine."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s], [s, c]])

gauche = rotation(np.pi / 3) @ rotation(np.pi / 6)
droite = rotation(np.pi / 2)
print(np.allclose(gauche, droite))
print(np.round(gauche, 10))

# allclose et non == : à cause des arrondis, les deux matrices ne sont pas
# égales bit à bit.
print(gauche == droite)

True
[[ 0. -1.]
 [ 1.  0.]]
[[False  True]
 [ True False]]


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [9]:
def determinant(A, tolerance=1e-14):
    """Déterminant de A par élimination de Gauss avec pivot partiel."""
    n = len(A)
    M = [list(ligne) for ligne in A]
    det = 1
    for k in range(n):
        p = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[p][k]) <= tolerance:
            return 0
        if p != k:
            # un échange de lignes change le signe du déterminant
            M[k], M[p] = M[p], M[k]
            det = -det
        det *= M[k][k]
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] -= facteur * M[k][j]
    return det

A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print(determinant(A), float(np.linalg.det(np.array(A))))

S = [[1, 2], [2, 4]]
print(determinant(S), float(np.linalg.det(np.array(S))))

-2.000000000000001 -2.0000000000000013
0 0.0


## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [10]:
from fractions import Fraction

def hilbert_exacte(n):
    return [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]

n = 12
H_exacte = hilbert_exacte(n)
b_exact = [sum(ligne) for ligne in H_exacte]

try:
    resoudre(H_exacte, b_exact)
except ValueError as erreur:
    print("premier essai :", erreur)

# Le dernier pivot vaut exactement 1/170392979877120, soit environ 5,9e-15 :
# il n'est pas nul, mais il passe sous le seuil de 1e-14 prévu pour le bruit
# d'arrondi des flottants. Avec des fractions il n'y a pas de bruit, le seul
# pivot à refuser est le zéro exact.
x_exact = resoudre(H_exacte, b_exact, tolerance=0)
print(x_exact)
print(all(v == 1 for v in x_exact))

premier essai : Système singulier.
[Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1)]
True


## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [11]:
def translation(tx, ty):
    """Matrice 3 x 3 de la translation de vecteur (tx, ty)."""
    return np.array([[1, 0, tx],
                     [0, 1, ty],
                     [0, 0, 1]])

def rotation_h(angle):
    """Matrice 3 x 3 de la rotation d'angle donné (en radians) autour de l'origine."""
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s, 0],
                     [s, c, 0],
                     [0, 0, 1]])

point = np.array([1, 0, 1])  # le point (1, 0) en coordonnées homogènes
T = translation(3, 1)
R = rotation_h(np.pi / 2)

# le produit se lit de droite à gauche : la matrice la plus proche du point agit en premier
print(np.round(R @ T @ point, 10)[:2])   # translation puis rotation
print(np.round(T @ R @ point, 10)[:2])   # rotation puis translation

[-1.  4.]
[3. 2.]


## L'IA vous le donne en trois secondes

**Système linéaire.** J'ai posé la question à un assistant IA, puis j'ai exécuté le code qu'il proposait pour vérifier ses affirmations.

Il propose `np.linalg.solve` d'emblée et met de lui-même en garde contre `inv(A) @ b`, « plus lent et moins précis ». La seconde question n'a pas été nécessaire.